# Setup

In [3]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/yelp.duckdb")

# Load and overview

In [4]:
con.sql("""
CREATE OR REPLACE TABLE business_raw AS
SELECT * FROM read_json_auto(
    '../data/yelp_academic_dataset_business.json',
    format='newline_delimited'
)""")

con.sql("DESCRIBE business_raw").df()

,column_name,column_type,null,key,default,extra
0,business_id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,address,VARCHAR,YES,None,None,None
3,city,VARCHAR,YES,None,None,None
4,state,VARCHAR,YES,None,None,None
5,postal_code,VARCHAR,YES,None,None,None
6,latitude,DOUBLE,YES,None,None,None
7,longitude,DOUBLE,YES,None,None,None
8,stars,DOUBLE,YES,None,None,None
9,review_count,BIGINT,YES,None,None,None


In [5]:
con.sql("""
SELECT business_id, name, city, state, stars, review_count, is_open, categories
FROM business_raw
LIMIT 5 """).df()

,business_id,name,city,state,stars,review_count,is_open,categories
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",Santa Barbara,CA,5.0,7,0,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,Affton,MO,3.0,15,1,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,Tucson,AZ,3.5,22,0,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,PA,4.5,13,1,"Brewpubs, Breweries, Food"


In [6]:
con.sql("SELECT COUNT(*) AS businesses FROM business_raw").df()

,businesses
0,150346


# Data quality checks

In [7]:
con.sql("SELECT COUNT(*) - COUNT(DISTINCT business_id) AS duplicate_ids FROM business_raw").df()

,duplicate_ids
0,0


In [8]:
con.sql("""
SELECT COUNT(*) AS total,
       COUNT(*) - COUNT(city)       AS missing_city,
       COUNT(*) - COUNT(categories) AS missing_categories,
       COUNT(*) - COUNT(latitude)   AS missing_lat,
       COUNT(*) - COUNT(stars)      AS missing_stars
FROM business_raw""").df()

,total,missing_city,missing_categories,missing_lat,missing_stars
0,150346,0,103,0,0


In [9]:
con.sql("""
SELECT MIN(latitude) AS min_lat, MAX(latitude) AS max_lat,
       MIN(longitude) AS min_lon, MAX(longitude) AS max_lon,
       SUM(CASE WHEN latitude = 0 AND longitude = 0 THEN 1 ELSE 0 END) AS zero_coords
FROM business_raw
""").df()

,min_lat,max_lat,min_lon,max_lon,zero_coords
0,27.555127,53.679197,-120.095137,-73.200457,0.0


In [10]:
con.sql("""
SELECT  MIN(stars) AS min_stars,
        MAX(stars) AS max_stars, 
        MIN(review_count) AS min_reviews, 
        MAX(review_count) AS max_reviews 
FROM business_raw""").df()

,min_stars,max_stars,min_reviews,max_reviews
0,1.0,5.0,5,7568


# Exploration

## Business by city

In [11]:
con.sql("SELECT COUNT(DISTINCT city || state) AS cities FROM business_raw").show()

┌────────┐
│ cities │
│ int64  │
├────────┤
│   1467 │
└────────┘



In [12]:
con.sql("""
SELECT city, state,
       COUNT(*) AS businesses,
       SUM(review_count) AS reviews
FROM business_raw
GROUP BY city, state
ORDER BY businesses DESC
LIMIT 15 """).show()

┌──────────────────┬─────────┬────────────┬─────────┐
│       city       │  state  │ businesses │ reviews │
│     varchar      │ varchar │   int64    │ int128  │
├──────────────────┼─────────┼────────────┼─────────┤
│ Philadelphia     │ PA      │      14567 │  936207 │
│ Tucson           │ AZ      │       9249 │  387239 │
│ Tampa            │ FL      │       9048 │  439450 │
│ Indianapolis     │ IN      │       7540 │  349228 │
│ Nashville        │ TN      │       6968 │  440988 │
│ New Orleans      │ LA      │       6208 │  621330 │
│ Reno             │ NV      │       5932 │  334555 │
│ Edmonton         │ AB      │       5054 │   98204 │
│ Saint Louis      │ MO      │       4827 │  244360 │
│ Santa Barbara    │ CA      │       3829 │  262853 │
│ Boise            │ ID      │       2937 │  101893 │
│ Clearwater       │ FL      │       2221 │   84190 │
│ Saint Petersburg │ FL      │       1663 │   76219 │
│ Metairie         │ LA      │       1643 │   61970 │
│ Sparks           │ NV     

## Top categories

In [13]:
con.sql("""
SELECT TRIM(category) AS category, COUNT(*) AS businesses
FROM (
    SELECT UNNEST(string_split(categories, ',')) AS category
    FROM business_raw
    WHERE categories IS NOT NULL
)
GROUP BY 1
ORDER BY 2 DESC
LIMIT 20 """).df()

,category,businesses
0,Restaurants,52268
1,Food,27781
2,Shopping,24395
3,Home Services,14356
4,Beauty & Spas,14292
5,Nightlife,12281
6,Health & Medical,11890
7,Local Services,11198
8,Bars,11065
9,Automotive,10773


## Stars and open/closed

In [14]:
con.sql("SELECT stars, COUNT(*) AS businesses FROM business_raw GROUP BY stars ORDER BY stars").df()

,stars,businesses
0,1.0,1986
1,1.5,4932
2,2.0,9527
3,2.5,14316
4,3.0,18453
5,3.5,26519
6,4.0,31125
7,4.5,27181
8,5.0,16307


In [15]:
con.sql("SELECT is_open, COUNT(*) AS businesses FROM business_raw GROUP BY is_open").df()

,is_open,businesses
0,0,30648
1,1,119698


In [16]:
con.sql("""
SELECT is_open,
       COUNT(*) AS businesses,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM business_raw
GROUP BY is_open """).df()

,is_open,businesses,pct
0,0,30648,20.4
1,1,119698,79.6


# Choosing the scope

## Restaurants in the candidate cities

In [17]:
con.sql("""
SELECT city, state,
       COUNT(*) AS restaurants,
       SUM(review_count) AS reviews,
       ROUND(AVG(review_count), 0) AS avg_reviews_per_business,
       SUM(is_open) AS open_restaurants
FROM business_raw
WHERE categories LIKE '%Restaurants%'
GROUP BY city, state
ORDER BY reviews DESC
LIMIT 10 """).df()

,city,state,restaurants,reviews,avg_reviews_per_business,open_restaurants
0,Philadelphia,PA,5852,665732.0,114.0,3525.0
1,New Orleans,LA,2259,465982.0,206.0,1391.0
2,Nashville,TN,2502,318548.0,127.0,1681.0
3,Tampa,FL,2960,293122.0,99.0,1964.0
4,Indianapolis,IN,2862,242024.0,85.0,1904.0
5,Tucson,AZ,2466,238851.0,97.0,1639.0
6,Reno,NV,1286,191003.0,149.0,796.0
7,Saint Louis,MO,1790,171777.0,96.0,957.0
8,Santa Barbara,CA,767,158247.0,206.0,446.0
9,Boise,ID,847,64366.0,76.0,601.0


## Decision

RAG scope: restaurants in New Orleans, LA (2,259 restaurants, about 466K reviews). It has the most reviews per restaurant (206), and the size keeps loading and embedding manageable. Philadelphia is larger (about 666K reviews) with less text per restaurant. The dashboard uses restaurants in all states. Closed restaurants are kept, with is_open as a column. The 103 businesses with no categories are excluded. Suburbs such as Metairie are separate cities in the data and are not included.

## Create the scoped table

In [18]:
# all restaurants, all states (dashboard)
con.sql("""
CREATE OR REPLACE TABLE business_restaurants AS
SELECT * FROM business_raw
WHERE categories LIKE '%Restaurants%' """)

# one city (RAG)
con.sql("""
CREATE OR REPLACE TABLE business_rag AS
SELECT business_id, name, city, state, latitude, longitude, stars, review_count, is_open, categories, hours, attributes
FROM business_restaurants
WHERE city = 'New Orleans' AND state = 'LA'
""")

con.sql("""
SELECT 'business_restaurants' AS table_name, COUNT(*) AS businesses, SUM(review_count) AS reviews FROM business_restaurants
UNION ALL
SELECT 'business_rag', COUNT(*), SUM(review_count) FROM business_rag """).df()



,table_name,businesses,reviews
0,business_restaurants,52268,4561279.0
1,business_rag,2259,465982.0


# Conclusion

## What I did
- Loaded the Yelp business file (150,346 businesses) into DuckDB.
- Checked the data quality, explored cities, categories and ratings, and chose the scope for the project.

## Data quality
- There are no duplicate business ids, so `business_id` is a safe key for joining other tables.
- City, latitude and stars have no missing values.
- 103 businesses (0.07%) have no categories. They are left out of the restaurant scope because I can't tell what type of business they are.

## Scope I chose
- **RAG system:** restaurants in New Orleans, LA. There are 2,259 restaurants and about 466K reviews. New Orleans has the most reviews per restaurant (206), and the size is small enough to run on my Mac.
- **Dashboard:** restaurants in all states.
- Closed restaurants are kept, and `is_open` stays as a column so I can filter them later.
- Nearby suburbs such as Metairie are separate cities in the data, so they are not included.

## Limits
- The review total (465,982) is the sum of the `review_count` column. The real number of reviews may be a little different, so I will check it when I load the review file.
- Some businesses are bars or cafes that are also tagged as restaurants.

In [19]:
con.close()